In [1]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import faiss
import numpy as np

c:\Users\LENOVO\anaconda3\envs\env3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FILE_DIR = 'D:/school/AI Project/RAG/dataset/'

In [3]:
df = pd.read_csv(FILE_DIR + 'arxiv_nlp.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101840 entries, 0 to 101839
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        101840 non-null  object
 1   title     101840 non-null  object
 2   authors   101840 non-null  object
 3   comments  64969 non-null   object
 4   abstract  101840 non-null  object
dtypes: object(5)
memory usage: 3.9+ MB


In [4]:
df.drop('comments', axis=1, inplace=True)

In [5]:
df['combined_text'] = 'title: ' + df['title'] + " | Abstract: " + df['abstract']

In [6]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)
df['vector'] = list(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 720.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3183/3183 [13:32<00:00,  3.92it/s]


In [11]:
# get embeddings dimension,. since i use all-MiniLM-L6-v2, the dimension is guaranteed 384.
dimension = embeddings.shape[1]

# baseline vector search
# search the vectors one by one, by storing them first into a data structure
# because the data is not that big (<1m), we can still use IndexFlatL2 search
# euclidean distance based to measure how far the distance between 2 vectors, return the smallest distance
index = faiss.IndexFlatL2(dimension)

In [12]:
# create a map with the key as vector, and paper id as value
# this is built on top of C++ space which makes it more efficient
index_with_ids = faiss.IndexIDMap(index)

# Create an array of integer IDs (FAISS requires int64)
paper_ids = np.arange(len(df)).astype('int64')

# index_with_ids.add_with_ids(vectors, paper_ids)
index_with_ids

<faiss.swigfaiss_avx2.IndexIDMap; proxy of <Swig Object of type 'faiss::IndexIDMapTemplate< faiss::Index > *' at 0x0000023D6CCB6D90> >

In [13]:
# Save the index to a file
faiss.write_index(index, "arxiv_nlp_index.faiss")

# Load it back later (even in a different script!)
loaded_index = faiss.read_index("arxiv_nlp_index.faiss")

In [16]:
len(df['combined_text'][0])

650